In [22]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score


In [10]:
outcomes = pd.read_csv('outcomes.csv')
projects = pd.read_csv('projects.csv')

In [11]:
binary_cols = ['fully_funded', 'at_least_1_teacher_referred_donor', 'great_chat',
               'three_or_more_non_teacher_referred_donors', 
               'one_non_teacher_referred_donor_giving_100_plus',
               'donation_from_thoughtful_donor', 'at_least_1_green_donation']


In [12]:
for col in binary_cols:
    outcomes[col] = outcomes[col].map({'t': 1, 'f': 0})

# Merge datasets
df = projects.merge(outcomes[['projectid', 'fully_funded']], on='projectid', how='inner')

In [14]:
drop_cols = ['projectid', 'teacher_acctid', 'schoolid', 'school_ncesid', 'school_city', 
             'school_zip', 'school_district', 'school_county', 'school_latitude', 
             'school_longitude']
df = df.drop(columns=drop_cols, errors='ignore')
df

,school_state,school_metro,school_charter,school_magnet,school_year_round,school_nlns,school_kipp,school_charter_ready_promise,teacher_prefix,teacher_teach_for_america,...,poverty_level,grade_level,fulfillment_labor_materials,total_price_excluding_optional_support,total_price_including_optional_support,students_reached,eligible_double_your_impact_match,eligible_almost_home_match,date_posted,fully_funded
0,IL,suburban,f,f,f,f,f,f,Mrs.,f,...,moderate poverty,Grades 3-5,30.0,444.36,522.78,7.0,f,f,2013-12-31,1
1,ID,urban,f,f,f,f,f,f,Mrs.,f,...,high poverty,Grades 3-5,30.0,233.24,274.40,30.0,f,f,2013-12-31,0
2,NH,suburban,f,f,f,f,f,f,Mrs.,f,...,moderate poverty,Grades 6-8,30.0,285.09,335.40,230.0,f,f,2013-12-31,0
3,VA,urban,f,f,f,f,f,f,Ms.,f,...,highest poverty,Grades PreK-2,30.0,232.94,274.05,18.0,f,f,2013-12-31,0
4,IL,urban,f,t,f,f,f,f,Mr.,f,...,highest poverty,Grades 6-8,30.0,513.41,604.01,70.0,t,f,2013-12-31,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619321,NY,urban,f,f,f,f,f,f,Ms.,f,...,highest poverty,Grades PreK-2,NaN,231.00,281.71,0.0,f,f,2002-09-17,1
619322,NY,urban,f,f,f,f,f,f,Mr.,f,...,highest poverty,Grades 9-12,NaN,1129.00,1376.83,0.0,f,f,2002-09-17,1
619323,NY,urban,f,t,f,f,f,f,Ms.,f,...,moderate poverty,Grades 3-5,NaN,125.00,152.44,0.0,f,f,2002-09-16,1
619324,NY,NaN,f,f,f,f,f,f,Ms.,f,...,highest poverty,Grades 9-12,NaN,125.00,152.44,0.0,f,f,2002-09-13,1


In [15]:
categorical_cols = [
    'school_state', 'school_metro', 'teacher_prefix', 'primary_focus_area',
    'primary_focus_subject', 'secondary_focus_area', 'secondary_focus_subject',
    'resource_type', 'poverty_level', 'grade_level',
    'school_charter', 'school_magnet', 'school_year_round',
    'school_nlns', 'school_kipp', 'school_charter_ready_promise',
    'teacher_teach_for_america', 'teacher_ny_teaching_fellow',
    'eligible_double_your_impact_match', 'eligible_almost_home_match'
]

In [16]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df.fillna(0, inplace=True)

In [17]:
df['date_posted'] = pd.to_datetime(df['date_posted'])
df.sort_values('date_posted', inplace=True)
df.drop(columns=['date_posted'], inplace=True)

In [18]:
# -----------------------
# 3. Incremental Feature Selection
# -----------------------

In [19]:
split = int(0.8 * len(df))
train_df, test_df = df.iloc[:split], df.iloc[split:]

X_train_all, y_train = train_df.drop('fully_funded', axis=1), train_df['fully_funded']
X_test_all, y_test = test_df.drop('fully_funded', axis=1), test_df['fully_funded']

In [20]:
available_features = list(X_train_all.columns)
selected_features = []

# Baseline AUC (no features, random guessing)
best_auc = 0.5

In [23]:
print("\n--- Incremental Feature Selection (Random Forest) ---")
for feature in available_features:
    trial_features = selected_features + [feature]

    model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
    model.fit(X_train_all[trial_features], y_train)
    preds = model.predict_proba(X_test_all[trial_features])[:, 1]
    auc = roc_auc_score(y_test, preds)

    print(f"Testing feature: {feature:<40} | AUC: {auc:.4f}", end=' ')

    if auc >= best_auc:
        selected_features.append(feature)
        best_auc = auc
        print("✅ Added")
    else:
        print("❌ Not added")

print("\nFinal selected features (Random Forest):")
print(selected_features)


--- Incremental Feature Selection (Random Forest) ---
Testing feature: fulfillment_labor_materials              | AUC: 0.5000 ✅ Added
Testing feature: total_price_excluding_optional_support   | AUC: 0.6527 ✅ Added
Testing feature: total_price_including_optional_support   | AUC: 0.6530 ✅ Added
Testing feature: students_reached                         | AUC: 0.6533 ✅ Added
Testing feature: school_state_AL                          | AUC: 0.6532 ❌ Not added
Testing feature: school_state_AR                          | AUC: 0.6535 ✅ Added
Testing feature: school_state_AZ                          | AUC: 0.6531 ❌ Not added
Testing feature: school_state_CA                          | AUC: 0.6559 ✅ Added
Testing feature: school_state_CO                          | AUC: 0.6558 ❌ Not added
Testing feature: school_state_CT                          | AUC: 0.6560 ✅ Added
Testing feature: school_state_DC                          | AUC: 0.6561 ✅ Added
Testing feature: school_state_DE                     

In [24]:
selected_features_dt = []
best_auc_dt = 0.5

print("\n--- Incremental Feature Selection (Decision Tree) ---")
for feature in available_features:
    trial_features = selected_features_dt + [feature]

    dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
    dt_model.fit(X_train_all[trial_features], y_train)
    preds_dt = dt_model.predict_proba(X_test_all[trial_features])[:, 1]
    auc_dt = roc_auc_score(y_test, preds_dt)

    print(f"Testing feature: {feature:<40} | AUC: {auc_dt:.4f}", end=' ')

    if auc_dt >= best_auc_dt:
        selected_features_dt.append(feature)
        best_auc_dt = auc_dt
        print("✅ Added")
    else:
        print("❌ Not added")

print("\nFinal selected features (Decision Tree):")
print(selected_features_dt)



--- Incremental Feature Selection (Decision Tree) ---
Testing feature: fulfillment_labor_materials              | AUC: 0.5000 ✅ Added
Testing feature: total_price_excluding_optional_support   | AUC: 0.6509 ✅ Added
Testing feature: total_price_including_optional_support   | AUC: 0.6488 ❌ Not added
Testing feature: students_reached                         | AUC: 0.6485 ❌ Not added
Testing feature: school_state_AL                          | AUC: 0.6503 ❌ Not added
Testing feature: school_state_AR                          | AUC: 0.6505 ❌ Not added
Testing feature: school_state_AZ                          | AUC: 0.6509 ✅ Added
Testing feature: school_state_CA                          | AUC: 0.6534 ✅ Added
Testing feature: school_state_CO                          | AUC: 0.6534 ✅ Added
Testing feature: school_state_CT                          | AUC: 0.6534 ✅ Added
Testing feature: school_state_DC                          | AUC: 0.6534 ✅ Added
Testing feature: school_state_DE                 

In [25]:
# -----------------------
# 4. Final Model Evaluation
# -----------------------


In [26]:
final_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
final_rf.fit(X_train_all[selected_features], y_train)
final_preds_rf = final_rf.predict(X_test_all[selected_features])
final_auc_rf = roc_auc_score(y_test, final_preds_rf)

print("\nRandom Forest Final ROC-AUC:", round(final_auc_rf, 4))


Random Forest Final ROC-AUC: 0.552


In [27]:
final_dt = DecisionTreeClassifier(max_depth=10, random_state=42)
final_dt.fit(X_train_all[selected_features_dt], y_train)
final_preds_dt = final_dt.predict(X_test_all[selected_features_dt])
final_auc_dt = roc_auc_score(y_test, final_preds_dt)

print("Decision Tree Final ROC-AUC:", round(final_auc_dt, 4))

Decision Tree Final ROC-AUC: 0.5486
